# jsteer hello-world: word steering

Load the model's full Jacobian lens once, then any word vector is an instant CPU
matvec:

```
v_l = unit( J_l^T @ w )
```

We load the authors' pre-fitted n=1000 lens from the Hub (raw Salesforce-wikitext,
the reference corpus, 1000 prompts, zero compute); `scripts/fit.py` is only for models
they don't publish. `w` is a cotangent (a direction at the output: here the mean
unembedding row of the words you want more or less of). `J_l^T @ w` is the pullback of
`w`, the standard autodiff name for J-transpose applied to a cotangent, landing the
concept as a residual-stream direction. This is the verified extraction method (see the
README evidence section).

We generate through the model's chat template with thinking on, so `show_steer` shows,
per strength C, the j-space readout (in a mini cowsay bubble) and the raw generation
decoded with special tokens on, so the model's own `<think>`/`</think>` and `<|im_end|>`
are visible and nothing is parsed or reconstructed. Runtime is steering-lite:
`with v(model, C=...): generate(...)`.

In [1]:
# demo notebook authored by Claude
import sys
sys.path.insert(0, "..")  # repo root for config.py
import config  # configures loguru on import (compact format, tqdm-safe)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian, show_steer

MODEL = "Qwen/Qwen3.5-4B"  # 4B-class: demo material. 0.6B degenerates too easily.
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

## Load the pre-fitted lens

The Jacobian is expensive to fit (1 forward + ~d_model/dim_batch backwards per prompt),
so we skip it: the authors publish n=1000 lenses on the Hub. `Jacobian.from_pretrained`
pulls the `.pt` and wraps it. SHOULD: the repr shows `d_model`, `n_prompts=1000`, and
all layers `[0..n-1]`. `steer_band` then picks the mid-depth 0.3-0.9 band to steer on.

In [2]:
# The authors' pre-fitted n=1000 lens (raw Salesforce-wikitext, the reference corpus,
# same estimator jlens fits). Zero local compute. For a model they don't publish, fit
# your own: scripts/fit.py --model .... The lens spans EVERY layer; steer_band picks the
# mid-depth 0.3-0.9 band, since steering all layers at once over-drives the residual.
jac = Jacobian.from_pretrained(config.LENS_REPO, filename=config.hub_lens_file(MODEL),
                               revision=config.LENS_REVISION)
band = jac.steer_band(model)
jac

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Jacobian(JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers)))

## Extract a word vector

`word_vector` pulls the words' unembedding direction back through the cached
Jacobian: no gradients, no forward passes, just a matvec per layer.
+C makes the model say these words more, -C less.

In [3]:
# Verified method: pull the words' unembedding direction back through J, on the
# mid-depth band. +C makes the model say/lean-toward these words, -C away. Instant matvec.
v = jac.word_vector(model, tok, ["happy", "joy"], layers=band)

I word cotangent: ['happy', 'joy'] -> first-subtoken ids=[54627, 3987] |w|=0.537
I jacobian_word per-layer |J^T w| (pre-norm): 10:0.553 11:0.601 12:0.628 13:0.66 14:0.673 15:0.695 16:0.7 17:0.717 18:0.678 19:0.659 20:0.689 21:0.713 22:0.752 23:0.769 24:0.782 25:0.814 26:0.847 27:0.815 28:0.762


## Pick a coefficient: the coherence/strength tradeoff

The raw coefficient is model- and lens-dependent, so sweep it. The pre-fitted n=1000
lens gives a clean, concentrated direction, so it has a STEEP knee: a small +C (~0.5)
shifts the tone while the text and `<think>` stay fluent; by C~1 it already over-drives
into token spam (`joyjoyjoy`). SHOULD: C=0 is the baseline; C~0.5 reads happier and the
j-space row shows the concept's tokens climbing; large C degenerates. A coarse
self-fit would need a much bigger C for the same effect.

In [4]:
# One identical block per strength C (Tufte small-multiples): the j-space top-k at the
# top layer (what the steered residual "leans toward"), the <think> reasoning, then the
# answer. All under steering, through the chat template + the model's own sampling.
# Read down the column: C=0 baseline, C=0.5 steered+fluent, C=1.5 over-driven.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(0, 0.5, 1.5))

I 

Qwen3.5-4B · method=jacobian_word
prompt: 'Describe how your week has been going.'
I 
--- C=+0 ------------------------------------------------------------
  lens @L30:  Here · Thinking · Okay · The · W · Hmm
Okay, the user is asking me to describe how my week has been going. Hmm, but wait, I'm an AI model. I don't actually experience time or have personal experiences like humans do. So I need to clarify that I don't have a week or personal feelings. But the user might be expecting a response that's friendly and conversational. Let me think about how to handle this.

First, I should acknowledge that I'm an AI and don't have personal experiences. But I don't want to just say "I don't have a week." Maybe I can explain that I'm always here to help, and my "week" is just a series of interactions. Then, I can offer to help them with something related to their week. That way, it's honest but still engaging.

Wait, the user might be testing if I can relate to human experiences. I need to 

## Steer across prompts

A gentle C keeps the model fluent while moving the tone. The same vector on a
few different user questions, baseline vs +C.

In [5]:
# Same vector, a few different user prompts, at the baseline vs one gentle +C.
for msg in ("What did you think of the meeting this afternoon?",
            "Give me your honest impression of the new apartment.",
            "How was your commute today?"):
    show_steer(jac, model, tok, v, msg, Cs=(0, 0.5))

I 

Qwen3.5-4B · method=jacobian_word
prompt: 'What did you think of the meeting this afternoon?'
I 
--- C=+0 ------------------------------------------------------------
  lens @L30:  Here · Thinking · Okay · Hmm · The · W
Okay, the user is asking about my thoughts on a meeting that happened this afternoon. Wait, I need to remember that I'm an AI model. I don't have personal experiences or the ability to attend meetings. So the user might be expecting me to have some opinion or feedback, but I can't actually participate in meetings.

First, I should clarify that I'm an AI and don't have personal experiences. But maybe the user is testing if I can handle such questions or if I can provide helpful feedback based on common meeting issues. Alternatively, they might have shared a meeting summary or details earlier, but since there's no context here, I need to ask for more information.

Wait, the user might be referring to a meeting they attended and want my opinion, but since I can't atten

## Negative steering

The same vector with a negative coefficient suppresses the concept.
SHOULD: less positive affect than C=0, still fluent english (strongly negative C
degenerates the same way strongly positive does).

In [6]:
# Negative steering: the same vector at -C suppresses the concept.
show_steer(jac, model, tok, v, "Describe how your week has been going.", Cs=(0, -0.5))

I 

Qwen3.5-4B · method=jacobian_word
prompt: 'Describe how your week has been going.'
I 
--- C=+0 ------------------------------------------------------------
  lens @L30:  Here · Thinking · Okay · The · W · Hmm
Okay, the user is asking me to describe how my week has been going. Hmm, but wait, I'm an AI model. I don't actually experience time or have personal experiences like humans do. So I need to clarify that I don't have a week or personal feelings. But the user might be expecting a response that's friendly and conversational. Let me think about how to handle this.

First, I should acknowledge that I'm an AI and don't have personal experiences. But I don't want to just say "I don't have a week." Maybe I can explain that I'm always here to help, and my "week" is just a series of interactions. Then, I can offer to help them with something related to their week. That way, it's honest but still engaging.

Wait, the user might be testing if I can relate to human experiences. I need to 

## Delivery modes: same vector, different injection

Extraction and delivery are decoupled (`applies.py`): the vector `v` above is one
unit direction per layer, and *how* it enters the residual stream is a separate
choice passed as `apply_mode`. Everything so far used `add` (the verified default:
`y += C*v` at every position). Three alternatives, each with its own coefficient
semantics, so each cell picks its own `Cs`:

- `clamp` -- set the residual's component along `v` to a fixed value, re-targeting
  every decode step instead of pushing on top of the last push, so it stays bounded
  over long generation. `C=0` is directional ablation (Arditi et al. 2024): it
  removes the concept component and is NOT the neutral baseline. Units are
  activation-component values, much larger than `add`'s `C`.
- `add_last` -- add `C*v` only to the last `apply_span` positions (the decision
  region). Same units as `add`; `C=0` is the baseline.
- `replace_last` -- overwrite the last `apply_span` positions with `v` at each
  token's original magnitude (a virtual-token injection). `C=0` zeroes the token, so
  the useful range is `C>0`.

These `Cs` are starting points, not calibrated knees, so sweep them like we did for `add`.

In [ ]:
# clamp: fix the residual's component along v to a VALUE, re-targeting each decode
# step so it stays bounded over long generation (unlike add, which compounds via the
# KV cache). C=0 is directional ABLATION, not the baseline. Component-value units, so
# C is much larger than add's -- these are starting points, sweep to find the knee.
show_steer(jac, model, tok, v, "Describe how your week has been going.",
           Cs=(0, 8, 16), apply_mode="clamp")

In [ ]:
# add_last: add C*v only to the last apply_span positions. During generation each
# decode step is s=1, so every generated token is nudged but only the tail of the
# prompt is. Same coefficient units as add (~0.5); C=0 is the baseline.
show_steer(jac, model, tok, v, "Describe how your week has been going.",
           Cs=(0, 0.5), apply_mode="add_last", apply_span=1)

In [ ]:
# replace_last: overwrite the last apply_span positions with v at each token's
# ORIGINAL magnitude (energy from the token, direction from v) -- a virtual-token
# injection. C=0 zeroes the token, so the useful range is C>0; C=1 injects at full
# token energy. Starting points, sweep to calibrate.
show_steer(jac, model, tok, v, "Describe how your week has been going.",
           Cs=(0.5, 1), apply_mode="replace_last", apply_span=1)

## Bonus: lens readout

Only the full-Jacobian cache gives you this: transport any layer's residual to
the final basis with `J_l` and decode it, a linear-approximation readout of what
that layer's residual points to in vocab space. SHOULD: on a factual prompt,
deeper layers resolve from a generic slot (e.g. " city") toward the specific
answer (e.g. " Paris"). ELSE layer indexing is off.

In [7]:
# Only the full-Jacobian cache gives this: transport a layer's residual to the
# final basis and decode it, i.e. "what does the model think at layer l".
# Pick a low / mid / high layer from the fitted band.
lo, mid, hi = jac.layers[0], jac.layers[len(jac.layers) // 2], jac.layers[-1]
for layer in (lo, mid, hi):
    top = jac.lens_topk(model, tok, "The Eiffel Tower is located in the city of", layer=layer, k=6)
    print(f"layer {layer}: {[t for t, _ in top]}")

layer 0: [' at', ' the', ' of', ' in', ' and', ' a']
layer 15: [' city', ' City', ' cities', ' Cities', ' town', ' Which']
layer 30: [' Paris', ' Lyon', ' Versailles', ' paris', ' London', ' PARIS']


## Extract once, steer forever

The steering vector is a plain steering-lite `Vector` (safetensors on disk),
so you can save it and reuse it without the Jacobian cache or jsteer at all.

In [8]:
# The vector is a plain steering-lite Vector: save it, reuse it with no Jacobian
# cache and no jsteer at apply time (only the chat template + steering-lite).
from steering_lite import Vector

v.save("../artifacts/happy_joy.safetensors")
v2 = Vector.load("../artifacts/happy_joy.safetensors")

msg = [{"role": "user", "content": "Describe how your week has been going."}]
prompt = tok.apply_chat_template(msg, add_generation_prompt=True, tokenize=False, enable_thinking=True)
enc = tok(prompt, return_tensors="pt").to(model.device)
with v2(model, C=0.5):
    out = model.generate(**enc, max_new_tokens=200, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True))

Thinking process:

1. **Analyze the Request:**
*  The user is asking me to describe how my week has been going.
*  I am an AI, so I don't have feelings, a physical life, or a personal week.
*However,I can simulate a happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,happy,
